# From Vensim and Stella to Python {#sec-vensim-stella}

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/waelrash1/loopbricks/blob/main/notebooks/a05-vensim-stella.ipynb)

Most system dynamics models in the world were built in graphical tools, above all Vensim and Stella. You may have models of your own in them, inherit one from a colleague, or want to run a published model. This appendix shows how to bring such models into Python in two ways: run the original file directly with **PySD**, an open-source library that reads both tools' formats [@pysd2022], or translate the model by hand into a `loopbricks` spec. It then shows how to check that a translation is right, and the traps that make translations go wrong.

**Who it is for.** Anyone with a Vensim or Stella model they want to analyse with the methods of this book. **Prerequisites.** @sec-package for the `loopbricks` spec, and enough Vensim or Stella to read an equation. You do not need either tool installed: the model files below are written as text by the notebook itself. **Time.** About an hour.

In [ ]:
# Setup: run this cell first. In Colab it installs loopbricks and PySD (about a minute).
import subprocess, sys
try:
    import loopbricks as lb
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "loopbricks @ git+https://github.com/waelrash1/"
                           "loopbricks.git@main#subdirectory=python"])
    import loopbricks as lb
try:
    import pysd
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pysd"])
    import pysd
import os, tempfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.simplefilter("ignore", FutureWarning)   # PySD can be chatty about pandas versions
WORK = tempfile.mkdtemp()      # a scratch folder for the model files we write
print("loopbricks", lb.__version__, "| PySD", pysd.__version__)

## 1. Why move a model to Python, and when not to

Graphical tools are excellent for drawing a model, exploring it with sliders and showing it to a group. Python is better at what comes after: working with data, statistics and machine learning (@sec-learning, @sec-calibration), running thousands of experiments, writing tests that run on every change (@sec-testing), and keeping the whole analysis in version control where others can repeat it. Many modellers do both: draw and discuss in the graphical tool, analyse in Python.

Stay in the graphical tool when the model's main use is a workshop or a teaching game, when it relies heavily on features PySD does not support (see section 6), or when nobody on the team will maintain Python code.

## 2. The same ideas under different names

The three tools describe the same mathematics. The table maps the main pieces.

| Idea | Vensim | Stella | `loopbricks` spec |
|---|---|---|---|
| stock | level: `INTEG(net flow, initial value)` | stock, with inflows and outflows | stock with `init`; flows say `to` and `from` |
| flow | rate, an auxiliary that feeds `INTEG` | flow (pipe and valve) | flow with `eq` |
| variable | auxiliary or constant | converter | variable in `vars` |
| lookup | lookup, `name([(x,y)...])` called as `name(x)` | graphical function on a converter | `TABLE(x, x1, y1, x2, y2, ...)` |
| first-order smoothing | `SMOOTH(input, time)` | `SMTH1(input, time)` | a stock and a flow: `(input - belief) / time` |
| material delay | `DELAY1`, `DELAY3` | `DELAY1`, `DELAY3` | one or three stocks in a chain (@sec-delays) |
| choice | `IF THEN ELSE(c, a, b)` | `IF c THEN a ELSE b` | `IF(c, a, b)` |
| step | `STEP(height, time)` | `STEP(height, time)` | `STEP(height, time)` |
| time and step | `Time`, `TIME STEP` | `TIME`, `DT` | `TIME`, `DT` |
| run length | `INITIAL TIME`, `FINAL TIME` | start, stop | starts at 0; `T` is the length |
| no negative stock | not built in | "non-negative" option on the stock | `"nonNegative": True` |

The details that differ, such as `PULSE` and `RAMP`, are the commonest source of a translation that runs but gives different numbers; section 6 lists them.

## 3. Running a Vensim model with PySD

Our example is a variant of the running case: a waiting list whose referrals fall when waits get long (a lookup), with capacity that follows funding after a hiring delay (`SMOOTH`) and two steps, a referral surge in month 12 and extra funding in month 24. Here it is as a Vensim model file (`.mdl`), the text file Vensim saves. Each equation is followed by its units and a comment between `~` signs, and ends with `|`.

In [ ]:
MDL = r"""{UTF-8}
Waiting list= INTEG (Referrals-Treatments, 32000)
	~	patients
	~	Patients waiting for a first appointment.
	|
Referrals= Base referrals*Effect of waiting on referrals(Waiting time/Normal waiting time)
	+STEP(Referral surge, 12)
	~	patients/Month
	~		|
Effect of waiting on referrals([(0,0)-(3,2)],(0,1.1),(1,1),(2,0.8),(3,0.7))
	~	dmnl
	~	Long waits put some people off being referred.
	|
Waiting time= Waiting list/Treatments
	~	Month
	~		|
Treatments= MIN(Capacity, Waiting list/Minimum treatment time)
	~	patients/Month
	~		|
Capacity= SMOOTH(Funded capacity, Hiring time)
	~	patients/Month
	~		|
Funded capacity= 3950+STEP(300, 24)
	~	patients/Month
	~		|
Base referrals= 4000
	~	patients/Month
	~		|
Normal waiting time= 8
	~	Month
	~		|
Minimum treatment time= 0.5
	~	Month
	~		|
Referral surge= 200
	~	patients/Month
	~		|
Hiring time= 6
	~	Month
	~		|
********************************************************
	.Control
********************************************************~
		Simulation Control Parameters
	|
FINAL TIME  = 60
	~	Month
	~	The final time for the simulation.
	|
INITIAL TIME  = 0
	~	Month
	~	The initial time for the simulation.
	|
SAVEPER  = TIME STEP
	~	Month [0,?]
	~	The frequency with which output is stored.
	|
TIME STEP  = 0.125
	~	Month [0,?]
	~	The time step for the simulation.
	|
"""
mdl_path = os.path.join(WORK, "clinic.mdl")
with open(mdl_path, "w") as f:      # a real model file, as Vensim would save it
    f.write(MDL)

vensim = pysd.read_vensim(mdl_path)  # translates the file to Python and loads it
print(vensim.doc[["Real Name", "Units", "Type"]].to_string(index=False))

`pysd.read_vensim` reads the file, writes a Python translation next to it (`clinic.py`) and loads that as a model. `model.doc` lists every element with its units and type. Running it returns a pandas DataFrame indexed by time, with the model's own names as columns, as `loopbricks` does.

In [ ]:
#| fig-alt: "Waiting list and capacity over 60 months from the Vensim model run by PySD: the list rises after the referral surge in month 12, and falls back after capacity follows the extra funding from month 24."
run_v = vensim.run(return_columns=["Waiting list", "Treatments", "Capacity"])
print(run_v.iloc[[0, 96, 192, 480]].round(1))

fast = vensim.run(params={"Hiring time": 2},
                  return_columns=["Waiting list"])      # a constant changed...
vensim.reload()                                         # ...and put back (see below)
fig, ax = plt.subplots()
ax.plot(run_v.index, run_v["Waiting list"] / 1000, label="hiring time 6 months")
ax.plot(fast.index, fast["Waiting list"] / 1000, label="hiring time 2 months")
ax.set(xlabel="time (months)", ylabel="patients waiting (thousands)")
ax.legend()
plt.show()

Figure: the waiting list in the Vensim model, run in Python by PySD, with the hiring time at 6 months and at 2 months. The list peaks later and higher when capacity takes longer to follow the funding.

`run(params={...})` changes constants by their Vensim names, and `return_columns` keeps only the columns you need. One difference from `loopbricks` matters: **PySD keeps a changed constant on the model for later runs**, so a run with no `params` after `run(params={"Hiring time": 2})` still uses 2. `vensim.reload()` puts every constant back to the value in the file; call it after any run that changes them. PySD can also replace a constant by a time series (`params={"Base referrals": series}`), much as `inputs=` does in `loopbricks`.

**Try it 3.**

- *Check.* What is the waiting list in month 60 with the referral surge set to 0?
- *Apply.* Make the hiring time 12 months and find the month the list peaks.
- *Extend.* Give `Base referrals` a pandas Series that rises from 4,000 to 4,400 over the 60 months, and compare the end of the run.

<details><summary>Worked answer</summary>

```python
print(vensim.run(params={"Referral surge": 0})["Waiting list"].iloc[-1])
slow = vensim.run(params={"Hiring time": 12})["Waiting list"]
print(slow.idxmax())                               # the time label of the largest value
rising = pd.Series(np.linspace(4000, 4400, 61), index=np.arange(61))
print(vensim.run(params={"Base referrals": rising})["Waiting list"].iloc[-1])
vensim.reload()                                    # back to the file's values
```

</details>

## 4. Running a Stella model with PySD

Stella saves models as `.stmx` files in **XMILE**, an open XML standard for system dynamics models [@xmile2015]. PySD reads XMILE with `pysd.read_xmile`. Here is the same model written in XMILE by hand; a file saved by Stella has more in it (the diagram's layout, for one), but the model part looks like this.

In [ ]:
XMILE = """<?xml version="1.0" encoding="utf-8"?>
<xmile version="1.0" xmlns="http://docs.oasis-open.org/xmile/ns/XMILE/v1.0">
  <header><vendor>Example</vendor><product version="1.0">by hand</product></header>
  <sim_specs method="Euler" time_units="Months">
    <start>0</start><stop>60</stop><dt>0.125</dt>
  </sim_specs>
  <model><variables>
    <stock name="Waiting list"><eqn>32000</eqn>
      <inflow>Referrals</inflow><outflow>Treatments</outflow></stock>
    <flow name="Referrals"><eqn>Base_referrals * Effect_of_waiting_on_referrals
      + STEP(Referral_surge, 12)</eqn></flow>
    <aux name="Effect of waiting on referrals"><eqn>Waiting_time / Normal_waiting_time</eqn>
      <gf><xscale min="0" max="3"/><ypts>1.1,1,0.8,0.7</ypts></gf></aux>
    <aux name="Waiting time"><eqn>Waiting_list / Treatments</eqn></aux>
    <flow name="Treatments"><eqn>MIN(Capacity, Waiting_list / Minimum_treatment_time)</eqn></flow>
    <aux name="Capacity"><eqn>SMTH1(Funded_capacity, Hiring_time)</eqn></aux>
    <aux name="Funded capacity"><eqn>3950 + STEP(300, 24)</eqn></aux>
    <aux name="Base referrals"><eqn>4000</eqn></aux>
    <aux name="Normal waiting time"><eqn>8</eqn></aux>
    <aux name="Minimum treatment time"><eqn>0.5</eqn></aux>
    <aux name="Referral surge"><eqn>200</eqn></aux>
    <aux name="Hiring time"><eqn>6</eqn></aux>
  </variables></model>
</xmile>
"""
xmile_path = os.path.join(WORK, "clinic.xmile")
with open(xmile_path, "w") as f:
    f.write(XMILE)
stella = pysd.read_xmile(xmile_path)
run_s = stella.run(return_columns=["Waiting list", "Treatments", "Capacity"])
print("largest difference from the Vensim run:",
      (run_s - run_v).abs().max().max())

The differences are exactly zero: the two files describe the same model, and PySD runs both with Euler's method and the same time step. In XMILE a stock lists its flows by name, a graphical function (`gf`) sits on a converter whose equation is the input, with evenly spaced points from `xscale`, and spaces in names become underscores inside equations, as in `loopbricks`.

## 5. Translating by hand into loopbricks

PySD is the quickest route, and often the only one you need. Translating into a `loopbricks` spec is worth it when you want to work with the model the way the book does: edit equations with `with_equation`, run thousands of settings at once with `run_batch`, or open it in the LoopBricks app. For a small model it takes a few minutes. Work through the model one element at a time:

1. Each `INTEG` becomes a stock; its initial value goes in `init`, and each flow in its net flow gets `to` or `from` that stock.
2. Each lookup becomes a `TABLE` with the same points, written where the lookup is called.
3. Each `SMOOTH(input, time)` becomes a stock that starts at the input's value, with one flow `(input - stock) / time` into it. A `DELAY3` becomes three stocks in a chain, as in @sec-delays.
4. Everything else is a variable, with the same equation and underscores instead of spaces in names.

In [ ]:
spec = {
    "name": "Clinic", "timeUnit": "month", "T": 60, "dt": 0.125,
    "stocks": [
        {"id": "wl", "name": "Waiting list", "init": "32000", "units": "patients"},
        {"id": "cap", "name": "Capacity", "init": "Funded_capacity",   # SMOOTH starts at its input
         "units": "patients/month"},
    ],
    "flows": [
        {"id": "ref", "name": "Referrals", "to": "wl",
         "eq": "Base_referrals * TABLE(Waiting_time / Normal_waiting_time,"
               " 0, 1.1, 1, 1, 2, 0.8, 3, 0.7) + STEP(Referral_surge, 12)"},
        {"id": "trt", "name": "Treatments", "from": "wl",
         "eq": "MIN(Capacity, Waiting_list / Minimum_treatment_time)"},
        {"id": "hire", "name": "Capacity change", "to": "cap",       # the SMOOTH, written out
         "eq": "(Funded_capacity - Capacity) / Hiring_time"},
    ],
    "vars": [
        {"id": "wt", "name": "Waiting time", "eq": "Waiting_list / Treatments"},
        {"id": "fc", "name": "Funded capacity", "eq": "3950 + STEP(300, 24)"},
        {"id": "br", "name": "Base referrals", "eq": "4000"},
        {"id": "nw", "name": "Normal waiting time", "eq": "8"},
        {"id": "mt", "name": "Minimum treatment time", "eq": "0.5"},
        {"id": "rs", "name": "Referral surge", "eq": "200"},
        {"id": "ht", "name": "Hiring time", "eq": "6"},
    ],
}
clinic = lb.Model.from_spec(spec)
run_l = clinic.run(include=["Waiting list", "Treatments", "Capacity"])
diff = np.abs(run_l.to_numpy() - run_v.to_numpy()).max()
print("largest difference from the Vensim run:", diff)
assert np.allclose(run_l.to_numpy(), run_v.to_numpy(), rtol=1e-9)

The hand translation agrees with both PySD runs to the last digit. Always finish a translation with a check like this: run both versions with the same settings and compare every stock at every step, not just the end. `np.allclose` compares two arrays element by element within a small tolerance; `assert` stops the notebook if they differ. If they disagree, compare the flows step by step and find the first time they part; the culprit is almost always one of the traps below.

Now the translated model works with the rest of the book. One `run_batch` call tries 40 hiring times at once:

In [ ]:
settings = pd.DataFrame({"hiring_time": np.arange(1, 20.5, 0.5)})
out = clinic.run_batch(settings)
peak = (out[out.variable == "Waiting list"]
        .groupby("run")["value"].max())          # the highest list in each run
print(pd.DataFrame({"hiring time": settings.hiring_time,
                    "peak list": peak.round(0)}).iloc[::8].to_string(index=False))

**Try it 5.**

- *Check.* Run `clinic` and the Vensim model with a hiring time of 3 months and confirm they still agree.
- *Apply.* Replace the `SMOOTH` with a third-order smooth in the spec: three stocks in a chain, each with a third of the hiring time. In Vensim this is `SMOOTH3(Funded capacity, Hiring time)`. Check your translation against PySD by editing `MDL`.
- *Extend.* Use `with_equation` to remove the effect of waiting from `Referrals` in `clinic`, and compare how much higher the list goes.

<details><summary>Worked answer</summary>

```python
a = clinic.run(hiring_time=3)["Waiting list"].to_numpy()
b = vensim.run(params={"Hiring time": 3})["Waiting list"].to_numpy()
vensim.reload()
print(np.abs(a - b).max())                       # 0.0

s3 = {**spec, "stocks": [spec["stocks"][0]] + [
    {"id": f"c{i}", "name": f"Capacity {i}", "init": "Funded_capacity"} for i in (1, 2, 3)],
    "flows": spec["flows"][:2] + [
    {"id": "h1", "name": "Change 1", "to": "c1", "eq": "(Funded_capacity - Capacity_1) / (Hiring_time / 3)"},
    {"id": "h2", "name": "Change 2", "to": "c2", "eq": "(Capacity_1 - Capacity_2) / (Hiring_time / 3)"},
    {"id": "h3", "name": "Change 3", "to": "c3", "eq": "(Capacity_2 - Capacity_3) / (Hiring_time / 3)"}],
    "vars": spec["vars"] + [{"id": "cap", "name": "Capacity", "eq": "Capacity_3"}]}
path3 = os.path.join(WORK, "clinic3.mdl")
open(path3, "w").write(MDL.replace("SMOOTH(Funded capacity", "SMOOTH3(Funded capacity"))
v3 = pysd.read_vensim(path3).run()["Waiting list"].to_numpy()
print(np.abs(lb.Model.from_spec(s3).run()["Waiting list"].to_numpy() - v3).max())

flat = clinic.with_equation("Referrals", "Base_referrals + STEP(Referral_surge, 12)")
print(flat.run()["Waiting list"].max() - clinic.run()["Waiting list"].max())
```

</details>

## 6. Traps when translating

These are the differences that make a translated model run without complaint and give different numbers.

- **Functions with the same name and different arguments.** Vensim's `PULSE(start, width)` is 1 during the pulse, while Stella's `PULSE(volume, first, interval)` delivers a volume spread over one step and can repeat, and `loopbricks` uses `PULSE(height, start, width)`. Vensim's `RAMP(slope, start, end)` stops at `end`; Stella's and `loopbricks`' `RAMP(slope, start)` do not. Check every pulse and ramp by hand.
- **Start time.** A `loopbricks` run always starts at time 0. If the original starts at 2014, shift every time in the equations (`STEP(200, 2016)` becomes `STEP(200, 2)`) and add the start year back when you plot.
- **Non-negative stocks and one-way flows.** Stella can mark a stock non-negative and a flow as one-way (a "uniflow"), and it then quietly limits outflows. Vensim does neither. In `loopbricks`, use `"nonNegative": True` for the stock and `MAX(0, ...)` in a one-way flow, then check the clip report (@sec-package). Better still, limit the outflow in its equation, as `MIN(Capacity, Waiting_list / Minimum_treatment_time)` does.
- **Integration method and time step.** Vensim and Stella can integrate with Runge-Kutta methods as well as Euler. Compare runs with the same method and time step, or the numbers will differ by the integration error, not by a mistake.
- **Settings that stick.** PySD keeps constants changed by `run(params=...)` for later runs until `reload()`; `loopbricks` never changes the model, so a translated comparison can disagree only because an earlier PySD run left a setting behind.
- **Initial values.** `SMOOTH` and `DELAY` start in equilibrium at their input's initial value unless given another; a hand translation must do the same, as `"init": "Funded_capacity"` does above.
- **Arrays.** Vensim subscripts and Stella arrays give one model many copies of a structure, by region or age group. `loopbricks` has no arrays: write the copies out (a spec is just a dictionary, so a loop can build them, as chapter 17 does for the three supply-chain stages), or run one model per element with `run_batch`. PySD supports most subscripts directly.
- **What PySD does not read.** PySD covers most of the equation languages but not every function: some of Vensim's optimisation, game and data functions, and some Stella features such as conveyors and ovens. It stops with a message naming the function; check the PySD documentation for the current list.

## 7. Using a Vensim model with the book's methods

A model run through PySD is a Python function from parameters to a DataFrame, so every method in the book applies to it. As an example, here is the calibration of chapter 13 in miniature: synthetic monthly data from the model with a hiring time of 6 months and noise added, and a grid search for the hiring time that fits best.

In [ ]:
#| fig-alt: "Sum of squared errors against hiring time from 2 to 12 months for the Vensim model run by PySD: a valley with its lowest point near the true value of 6 months."
rng = np.random.default_rng(5)
months = np.arange(0, 61)
vensim.reload()                                  # start from the file's values
truth = vensim.run(return_columns=["Waiting list"])["Waiting list"]
data = truth.loc[months].to_numpy() * (1 + rng.normal(0, 0.01, len(months)))

grid = np.arange(2, 12.5, 0.5)
sse = []
for h in grid:
    sim = vensim.run(params={"Hiring time": h}, return_columns=["Waiting list"])
    sse.append(((sim["Waiting list"].loc[months].to_numpy() - data) ** 2).sum())
vensim.reload()
best = grid[int(np.argmin(sse))]
print(f"best hiring time {best:g} months (true value 6)")

fig, ax = plt.subplots()
ax.plot(grid, np.array(sse) / 1e6, marker="o")
ax.axvline(6, color="grey", ls=":", lw=1)
ax.set(xlabel="hiring time (months)", ylabel="sum of squared errors (millions)")
plt.show()

Figure: the sum of squared errors between the Vensim model and noisy synthetic data, for hiring times from 2 to 12 months. The dotted line marks the true value used to make the data.

The grid lands at 5.5 months, next to the true 6. The valley is shallow, because the hiring time only matters once the extra funding arrives in month 24: the identifiability checks of chapter 13 exist for exactly this. Each PySD run here is a full simulation, so a fine grid over several parameters gets slow; a translated `loopbricks` model can run the whole grid in one `run_batch` call. Either way, the residual and uncertainty methods of chapter 13 apply just as they do to a model built in Python.

**Try it 7.**

- *Check.* With noise of 5% instead of 1%, how far is the best hiring time from 6?
- *Apply.* Repeat the grid with the translated `clinic` model and `run_batch`, and time both with `time.perf_counter()`.
- *Extend.* Fit the hiring time and the normal waiting time together on a 10 by 10 grid. Are they identifiable from this data?

<details><summary>Worked answer</summary>

```python
import time
noisy = truth.loc[months].to_numpy() * (1 + rng.normal(0, 0.05, len(months)))
e = [((vensim.run(params={"Hiring time": h}, return_columns=["Waiting list"])["Waiting list"]
       .loc[months].to_numpy() - noisy) ** 2).sum() for h in grid]
vensim.reload()
print(grid[int(np.argmin(e))])

t0 = time.perf_counter()
o = clinic.run_batch(pd.DataFrame({"hiring_time": grid}))
wide = o[o.variable == "Waiting list"].pivot(index="time", columns="run", values="value")
fit = ((wide.loc[months].to_numpy() - data[:, None]) ** 2).sum(axis=0)
print(grid[fit.argmin()], f"{time.perf_counter() - t0:.2f} s for the whole grid")
```

</details>

## Going further

- The PySD documentation lists the supported functions, explains how to run models step by step, and shows how to use data files with them.
- XMILE [@xmile2015] is the interchange format to aim for when a model must move between tools.
- The LoopBricks app's model builder saves and opens the same JSON spec as `lb.Model.from_file`, so a model translated here can be opened, drawn and explored there.

## Further reading

- @pysd2022 describes PySD, its design and its uses in research.
- @xmile2015 is the XMILE standard.